# Recreating Rehman, Shafi & Wickman (2009)

## "Hybrid Inflation Revisited in Light of WMAP5"
### arXiv:0901.4345, Phys. Rev. D 79, 103503

This notebook implements a step-by-step reconstruction of the paper's key results following the guide's modular structure.

**Units:** Reduced Planck mass $M_P = 1$ throughout (field values, masses, and $V^{1/4}$ quoted in units of $M_P = 2.435 \times 10^{18}$ GeV unless stated otherwise).

In [13]:
#This module implements, in order, the physics/derivation steps
#start by importing libraries & setting units

import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
 
Mp = 1.0  #reduced Planck mass, working units

## Stage 1: Slow-Roll Formalism Derivations

**GOAL:** Verify the slow-roll formalism on a toy potential $V(\phi) = \frac{1}{2}m^2\phi^2$ and confirm the known result $n_s \approx 1 - 2/N$ for large $N$. (results pasted in google doc)

**Derivation:**

For $V = \frac{1}{2}m^2\phi^2$:

- $\epsilon(\phi) = \frac{M_P^2}{2}\left(\frac{V'}{V}\right)^2 = \frac{2M_P^2}{\phi^2}$
- $\eta(\phi) = M_P^2\frac{V''}{V} = \frac{2M_P^2}{\phi^2}$
- $N(\phi) = \frac{1}{M_P^2}\int_{\phi_{\text{end}}}^{\phi} \frac{V}{V'} d\phi' = \frac{\phi^2 - \phi_{\text{end}}^2}{4M_P^2}$
- $\phi_{\text{end}}$ from $\epsilon(\phi_{\text{end}}) = 1 \Rightarrow \phi_{\text{end}} = \sqrt{2}M_P$
- $n_s = 1 - 6\epsilon + 2\eta = 1 - \frac{8M_P^2}{\phi^2} = 1 - \frac{2}{N} + \mathcal{O}(1/N^2)$

In [14]:
#MODULE 1: slow-roll formalism derivations
 
def epsilon_toy(phi, m):
    """Slow-roll epsilon for V = 1/2 m^2 phi^2, computed analytically:
    eps = (Mp^2/2)(V'/V)^2 = 2 Mp^2 / phi^2 for this potential (m cancels)."""
    return 2.0 * Mp**2 / phi**2
 
 
def eta_toy(phi, m):
    """Slow-roll eta for V = 1/2 m^2 phi^2: eta = Mp^2 V''/V = 2 Mp^2/phi^2."""
    return 2.0 * Mp**2 / phi**2
 
 
def N_toy_analytic(phi, phi_end):
    """N(phi) = (phi^2 - phi_end^2) / (4 Mp^2) for V = 1/2 m^2 phi^2.
 
    Derivation: N = (1/Mp^2) int_{phi_end}^{phi} V/V' dphi'
              = (1/Mp^2) int phi'/2 dphi' = (phi^2 - phi_end^2)/(4 Mp^2)
    with phi_end defined by epsilon(phi_end) = 1  =>  phi_end = sqrt(2) Mp.
    """
    return (phi**2 - phi_end**2) / (4.0 * Mp**2)
 
 
def phi_of_N_toy(N, phi_end):
    """Invert N(phi) analytically: phi(N) = sqrt(4 Mp^2 N + phi_end^2)."""
    return np.sqrt(4.0 * Mp**2 * N + phi_end**2)
 
 
def ns_toy(N, phi_end):
    """ns(N) for chaotic V=1/2 m^2 phi^2, exact (not just the N->infty limit)."""
    phi = phi_of_N_toy(N, phi_end)
    eps = epsilon_toy(phi, m=1.0)
    eta = eta_toy(phi, m=1.0)
    return 1.0 - 6.0 * eps + 2.0 * eta
 
 
def r_toy(N, phi_end):
    phi = phi_of_N_toy(N, phi_end)
    eps = epsilon_toy(phi, m=1.0)
    return 16.0 * eps
 
 
def module1_sanity_check(verbose=True):
    """Checkpoint 0/1 sanity harness: for large N, ns -> 1 - 2/N."""
    phi_end = np.sqrt(2.0) * Mp
    results = {}
    for N in [50, 60]:
        ns_exact = ns_toy(N, phi_end)
        ns_approx = 1.0 - 2.0 / N
        r_exact = r_toy(N, phi_end)
        results[N] = dict(ns_exact=ns_exact, ns_approx=ns_approx, r=r_exact)
        if verbose:
            print(f"N={N}: ns(exact)={ns_exact:.4f}  ns(1-2/N approx)={ns_approx:.4f}"
                  f"  r={r_exact:.4f}")
    return results

## Stage 2: Tree-Level Hybrid Potential and Critical Field [Uses Hybrid-inflation potential (Linde 1994; Dvali, Shafi & Schaefer 1994)]

**GOAL:** Define the tree-level hybrid inflation potential and compute the critical field $\phi_c$ where the $S=0$ (waterfall field) valley becomes unstable.

**The Standard Hybrid Inflation Potential:**

$$V(\phi, S) = \frac{\kappa}{4}(S^2 - M^2)^2 + \frac{1}{2}m^2\phi^2 + \frac{1}{2}g^2\phi^2 S^2$$

where:
- $\phi$: gauge-singlet inflaton
- $S$: "waterfall" field charged under the broken symmetry
- $M$: symmetry-breaking scale (sets overall inflationary energy density $\sim \kappa M^4/4$)
- $g$: "hybrid" coupling tying $\phi$ to $S$
- $m$: inflaton soft mass

**Critical Field:** Along the $S=0$ valley, the effective mass-squared of $S$ is:

$$m_S^2(\phi) = \frac{\partial^2 V}{\partial S^2}\bigg|_{S=0} = -\kappa M^2 + g^2\phi^2$$

The valley becomes unstable when $m_S^2(\phi_c) = 0$, giving:

$$\phi_c = \frac{\sqrt{\kappa}M}{g}$$

**Tree-Level Single-Field Potential:** Along $S=0$:

$$V_0(\phi) = \frac{\kappa}{4}M^4 + \frac{1}{2}m^2\phi^2$$

which is nearly flat (the "hybrid plateau") with a small $m^2\phi^2$ correction.

In [15]:

# MODULE 2: Tree-level hybrid potential V(phi, S) and the critical field
#   V(phi, S) = (kappa/4)(S^2 - M^2)^2 + (1/2) m^2 phi^2 + (1/2) g^2 phi^2 S^2
 
def V_full(phi, S, kappa, M, m, g):
    return (kappa / 4.0) * (S**2 - M**2)**2 + 0.5 * m**2 * phi**2 \
        + 0.5 * g**2 * phi**2 * S**2
 
 
def critical_field(kappa, M, g):
    """Return the critical field value where the S=0 valley becomes unstable."""
    return np.sqrt(kappa) * M / g
 
def mS2_along_valley(phi, kappa, M, g):
    """Effective (mass)^2 of the waterfall field S along the S=0 valley."""
    return -kappa * M**2 + g**2 * phi**2
 
 
def module2_checkpoint(kappa=1.0, M=1e-2, g=1.0, verbose=True):
    """Check the critical field and the effective mass-squared along the S=0 valley."""
    return np.sqrt(kappa) * M / g
 
 
def mS2_along_valley(phi, kappa, M, g):
    """Effective (mass)^2 of the waterfall field S along the S=0 valley."""
    return -kappa * M**2 + g**2 * phi**2
 
 
def module2_checkpoint(kappa=1.0, M=1e-2, g=1.0, verbose=True):
    phi_c = critical_field(kappa, M, g)
    if verbose:
        print(f"phi_c = {phi_c:.6e} Mp  (S=0 valley unstable for phi < phi_c)")
        print(f"m_S^2(phi_c) = {mS2_along_valley(phi_c, kappa, M, g):.3e} "
              "(should be ~0)")
    return phi_c
 
 
#MODULE 2: tree-level single-field potential along S = 0
def V_tree(phi, kappa, M, m):
    """V0(phi) = (kappa/4) M^4 + 1/2 m^2 phi^2, i.e. the S=0 valley.
 
    This is "nearly flat, with M^4 setting the overall inflationary energy
    scale" exactly as the guide anticipates, with the m^2 phi^2/2 term as
    the "small correction" for sub-Planckian phi << M/ m-ish regime.
    """
    return (kappa / 4.0) * M**4 + 0.5 * m**2 * phi**2
 
def dV_tree(phi, m):
    return m**2 * phi
 
def d2V_tree(phi, m):
    return m**2 * np.full_like(np.asarray(phi, dtype=float), 1.0) * m**0

## Stage 3: Defining the Tree-Level Blue Tilt Problem

**GOAL:** Compute slow-roll observables at tree level and confirm $n_s > 1$ (blue tilt) for sub-Planckian $\phi$, reproducing the problem the paper aims to fix.

**For Tree-Level Potential $V_0(\phi) = \frac{\kappa}{4}M^4 + \frac{1}{2}m^2\phi^2$:**

$$\epsilon(\phi) = \frac{M_P^2}{2}\left(\frac{m^2\phi}{V_0(\phi)}\right)^2 \approx \frac{2M_P^2m^4\phi^2}{\kappa^2M^8} \ll 1$$

$$\eta(\phi) = M_P^2\frac{m^2}{V_0(\phi)} \approx \frac{4M_P^2m^2}{\kappa M^4} > 0$$

$$n_s = 1 - 6\epsilon + 2\eta \approx 1 + \frac{8M_P^2m^2}{\kappa M^4} > 1$$

The spectral index is **blue-tilted** ($n_s > 1$) for sub-Planckian $\phi$, in conflict with WMAP5's preference for a red tilt ($n_s < 1$).

In [16]:
# MODULE 3: tree-level slow-roll observables -> reproduces the blue-tilt
#problem the paper sets out to fix.
 
def epsilon_of_V(phi, Vfun, dVfun, h=1e-6):
    V = Vfun(phi)
    dV = dVfun(phi)
    return 0.5 * Mp**2 * (dV / V)**2
 
def eta_of_V(phi, Vfun, dVfun, h=1e-6):
    """eta = Mp^2 V''/V, V'' obtained by central finite difference of dVfun."""
    V = Vfun(phi)
    d2V = (dVfun(phi + h) - dVfun(phi - h)) / (2 * h)
    return Mp**2 * d2V / V
 
def ns_from_eps_eta(eps, eta):
    return 1.0 - 6.0 * eps + 2.0 * eta
 
def r_from_eps(eps):
    return 16.0 * eps
 
 
def module3_tree_level_blue_tilt(kappa=1.0, M=1e-2, m=1e-6, phi_test=1e-2,
                                  verbose=True):
    """Checkpoint 3: compute ns at tree level for representative parameters
    and confirm ns > 1 (blue tilt) for the S=0 valley alone."""
    Vfun = lambda p: V_tree(p, kappa, M, m)
    dVfun = lambda p: dV_tree(p, m)
    eps = epsilon_of_V(phi_test, Vfun, dVfun)
    eta = eta_of_V(phi_test, Vfun, dVfun)
    ns = ns_from_eps_eta(eps, eta)
    if verbose:
        print(f"Tree level at phi={phi_test:.3e}: eps={eps:.3e} eta={eta:.3e} "
              f"ns={ns:.4f}  (blue tilt if ns>1: {ns > 1})")
    return ns

## Stage 4: One-Loop Coleman-Weinberg Correction (Quantum Corrections)

**GOAL:** Implement the one-loop Coleman-Weinberg correction from Yukawa-coupled right-handed neutrinos, which creates the "hilltop" feature in the potential.

**The Paper's Central Mechanism:** The Yukawa coupling of the inflaton $\phi$ (or waterfall field $S$) to right-handed neutrinos $N$ gives $N$ a $\phi$-dependent mass:

$$m_N(\phi) = y\phi$$

Integrating out $N$ at one loop yields the Coleman-Weinberg contribution:

$$V_{\text{loop}}(\phi) = -\frac{N_{\text{dof}}}{64\pi^2} m_N(\phi)^4 \left[\ln\left(\frac{m_N(\phi)^2}{Q^2}\right) - \frac{3}{2}\right]$$

with $N_{\text{dof}} = 3$ for three generations of right-handed neutrinos.

**Why This Creates a Hilltop:** For $\phi > Q$, $V_{\text{loop}}(\phi)$ becomes large and negative. Adding this to the rising tree-level term $\frac{1}{2}m^2\phi^2$ creates a local maximum in $V(\phi)$.

**Near the maximum:** $\eta(\phi) \approx M_P^2 V''/V$ becomes **large and negative**, pulling $n_s$ down via the $+2\eta$ term, producing a red tilt ($n_s < 1$).

General one-loop Coleman-Weinberg contribution of a Yukawa-coupled Dirac fermion of field-dependent mass $M(\phi) = y\phi$, with $N_{\text{dof}}$ internal degrees of freedom (here $N_{\text{dof}} = 3$, one per right-handed neutrino generation), renormalized at scale $Q$: 

$$V_{\text{loop}}(\phi) = -\frac{N_{\text{dof}}}{64\pi^2} M(\phi)^4 \left[\ln\left(\frac{M(\phi)^2}{Q^2}\right) - \frac{3}{2}\right]$$

The overall minus sign is the standard fermion-loop sign in the Coleman-Weinberg potential (bosons contribute with a plus sign). Because $M(\phi)$ grows with $\phi$, this term becomes large and negative for $\phi > Q$, which added on top of the rising tree-level $\frac{1}{2}m^2\phi^2$ term, manufactures a local MAXIMUM in $V(\phi)$: the "hilltop" feature that is the abstract's headline result.

In [17]:
# MODULE 4: one-loop Coleman-Weinberg correction from RH neutrinos

def mN(phi, y):
    """Field-dependent right-handed-neutrino mass from the Yukawa coupling."""
    return y * phi


def V_loop(phi, y, phi_c):
    """One-loop Coleman-Weinberg term, renormalized at mu = y*phi_c so the
    log stays positive during inflation (phi < phi_c):
    V_loop = -A*phi^4*ln(y*phi/mu),  A = y^4/(16 pi^2)."""
    A = y**4 / (16.0 * np.pi**2)
    mu = y * phi_c
    phi_safe = np.where(np.abs(phi) < 1e-300, 1e-300, phi)
    L = np.log(y * phi_safe / mu)
    return -A * phi_safe**4 * L


def dV_loop(phi, y, phi_c):
    """Analytic first derivative of V_loop w.r.t. phi.
    d/dphi [-A phi^4 ln(y phi/mu)] = -A phi^3 (4L + 1), L = ln(y phi/mu)."""
    A = y**4 / (16.0 * np.pi**2)
    mu = y * phi_c
    phi_safe = np.where(np.abs(phi) < 1e-300, 1e-300, phi)
    L = np.log(y * phi_safe / mu)
    return -A * phi_safe**3 * (4.0 * L + 1.0)


def d2V_loop(phi, y, phi_c):
    """Analytic second derivative of V_loop w.r.t. phi:
    d^2/dphi^2 [-A phi^4 ln(y phi/mu)] = -A phi^2 (12L + 7), L = ln(y phi/mu)."""
    A = y**4 / (16.0 * np.pi**2)
    mu = y * phi_c
    phi_safe = np.where(np.abs(phi) < 1e-300, 1e-300, phi)
    L = np.log(y * phi_safe / mu)
    return -A * phi_safe**2 * (12.0 * L + 7.0)


def V_total(phi, kappa, M, m, y, phi_c):
    return V_tree(phi, kappa, M, m) + V_loop(phi, y, phi_c)

def module4_hilltop_check(kappa=4.0, M=1.517864e-3, m=3.712526e-7, y=2.070586e-3,
                           g=0.0454922, verbose=True):
    """Checkpoint 4 (defaults = Table 1 benchmark f=0.65, phi_P=0.25)."""
    phi_c = critical_field(kappa, M, g)
    phis = np.linspace(1e-6, 5e-1, 4000) * Mp
    Vt = V_total(phis, kappa, M, m, y, phi_c)
    Vfun, dVfun, d2Vfun = make_V_and_dV(kappa, M, m, y, phi_c)
    phi_max = find_hilltop(Vfun, dVfun, 1e-4 * Mp, 0.9 * Mp)
    has_max = phi_max is not None
    if verbose:
        print(f"Local maximum found: {has_max}"
              + (f"  at phi ~ {phi_max:.4e} Mp" if has_max else ""))
    return has_max, phis, Vt

## Stage 5: Full Numerical Slow-Roll Observables

**GOAL:** Compute $\epsilon(\phi)$, $\eta(\phi)$, $N(\phi)$, $n_s(\phi)$, and $r(\phi)$ numerically on the full potential $V_{\text{total}}(\phi) = V_{\text{tree}} + V_{\text{loop}}$.

**Key Numerical Steps:**

1. **Find the hilltop maximum** $\phi_{\text{max}}$ by locating where $V'(\phi) = 0$.
2. **Find the end of inflation** $\phi_{\text{end}}$ as the **minimum** of:
   - $\phi_c$: the waterfall trigger field (from Module 2)
   - $\phi_{\epsilon=1}$: where $\epsilon(\phi) = 1$
3. **Compute number of e-folds:**

   $$N(\phi) = \frac{1}{M_P^2}\int_{\phi_{\text{end}}}^{\phi} \frac{V(\phi')}{V'(\phi')} d\phi'$$

4. **Solve for $\phi_*$** such that $N(\phi_*) = N_{\text{target}}$ (typically 50-60 e-folds).
5. **Compute observables:**

   $$n_s = 1 - 6\epsilon(\phi_*) + 2\eta(\phi_*)$$
   $$r = 16\epsilon(\phi_*)$$

In [18]:
#MODULE 5: full numerical slow-roll observables on V_total

def make_V_and_dV(kappa, M, m, y, phi_c):
    """Return callables V(phi), dV(phi), d2V(phi) for the full tree +
    one-loop potential, using ANALYTIC derivatives (see dV_loop/d2V_loop
    above). Earlier versions of this function used finite differences, but
    phi spans many orders of magnitude in this model and naive
    double-finite-differencing (needed for eta = V''/V) amplifies floating
    point noise catastrophically near the hilltop, where V' is tiny by
    construction. Analytic derivatives avoid this entirely.
    """
    def Vfun(phi):
        return V_total(phi, kappa, M, m, y, phi_c)

    def dVfun(phi):
        return dV_tree(phi, m) + dV_loop(phi, y, phi_c)

    def d2Vfun(phi):
        return m**2 + d2V_loop(phi, y, phi_c)

    return Vfun, dVfun, d2Vfun


def find_hilltop(Vfun, dVfun, phi_lo, phi_hi, n=20000):
    """Locate the genuine hilltop maximum of V(phi) in [phi_lo, phi_hi] by
    scanning for the LAST sign change of dV/dphi (+ -> -) on a log-spaced
    grid, then refining with brentq. Starting the scan away from phi=0
    avoids spurious numerical-derivative noise where V is extremely flat.
    """
    phis = np.geomspace(phi_lo, phi_hi, n)
    dVs = np.array([dVfun(p) for p in phis])
    sign_changes = np.where(np.diff(np.sign(dVs)) < 0)[0]
    if len(sign_changes) == 0:
        return None
    idx = sign_changes[-1]  # largest phi at which V turns over = true hilltop
    lo, hi_ = phis[idx], phis[idx + 1]
    return brentq(dVfun, lo, hi_)


def epsilon(phi, Vfun, dVfun):
    return 0.5 * Mp**2 * (dVfun(phi) / Vfun(phi))**2


def eta(phi, Vfun, dVfun, d2Vfun):
    return Mp**2 * d2Vfun(phi) / Vfun(phi)


def find_phi_end(Vfun, dVfun, phi_lo, phi_hi):
    """Solve epsilon(phi_end) = 1 for phi_end in (phi_lo, phi_hi)."""
    f = lambda p: epsilon(p, Vfun, dVfun) - 1.0
    return brentq(f, phi_lo, phi_hi)


def efolds_integrand(phi, Vfun, dVfun):
    return Vfun(phi) / (Mp**2 * dVfun(phi))


def N_efolds(phi, phi_end, Vfun, dVfun):
    val, _ = quad(efolds_integrand, phi_end, phi, args=(Vfun, dVfun),
                   limit=400, epsabs=1e-10, epsrel=1e-8)
    return val


def solve_phi_star(N_target, phi_end, Vfun, dVfun, phi_lo, phi_hi):
    """Find phi* such that N_efolds(phi*, phi_end) = N_target."""
    f = lambda p: N_efolds(p, phi_end, Vfun, dVfun) - N_target
    return brentq(f, phi_lo, phi_hi, xtol=1e-14, rtol=1e-12)


def module5_observables(kappa=4.0, M=1.517864e-3, m=3.712526e-7, y=2.070586e-3,
                         g=0.0454922, N_target=56.3, verbose=True):
    """Checkpoint 5 (defaults = Table 1 benchmark f=0.65, phi_P=0.25:
    should reproduce ns~0.989, r~1.5e-4 -- see Module 6 for the check)."""
    phi_c = critical_field(kappa, M, g)
    Vfun, dVfun, d2Vfun = make_V_and_dV(kappa, M, m, y, phi_c)
    phi_max = find_hilltop(Vfun, dVfun, 1e-4 * Mp, 0.9 * Mp)
    if phi_max is None:
        raise RuntimeError("no hilltop found for these parameters")
    try:
        phi_eps1 = find_phi_end(Vfun, dVfun, 1e-6 * Mp, phi_max * 0.999)
    except ValueError:
        phi_eps1 = np.inf
    phi_end = min(phi_c, phi_eps1)
    if not (0 < phi_end < phi_max):
        raise RuntimeError("phi_end not sensibly below the hilltop; "
                            "adjust kappa/M/g so phi_c < phi_max")
    phi_star = solve_phi_star(N_target, phi_end, Vfun, dVfun,
                               phi_end * 1.0001, phi_max * 0.9999)
    eps = epsilon(phi_star, Vfun, dVfun)
    et = eta(phi_star, Vfun, dVfun, d2Vfun)
    ns = ns_from_eps_eta(eps, et)
    r = r_from_eps(eps)
    if verbose:
        print(f"phi_max(hilltop)={phi_max:.4e} Mp  phi_c={phi_c:.4e} Mp  "
              f"phi_end={phi_end:.4e} Mp  phi*={phi_star:.4e} Mp")
        print(f"eps={eps:.3e}  eta={et:.3e}  ns={ns:.4f}  r={r:.3e}")
    return dict(phi_max=phi_max, phi_c=phi_c, phi_end=phi_end,
                phi_star=phi_star, eps=eps, eta=et, ns=ns, r=r)

In [19]:
#MODULE 6 : validate the loop-corrected potential against Rehman, Shafi &
# Wickman (2009) Table 1, using their OWN published (f, phi_P) benchmark
# points -- not an arbitrary parameter grid.
#
# Conversion notes (all traceable to the paper's text):
#   - Their tree level is V0 = kappa_paper^2 * M^4 (Eq. 1); ours is
#     V0 = (kappa/4) * M_code^4. Fixing kappa=4 makes M_code = V0^(1/4)
#     directly, removing the kappa-convention ambiguity.
#   - phi_c/phi0 = f_c/f (dimensionless ratio, convention-independent).
#   - A = y^4/(16 pi^2)  =>  y = (16 pi^2 A)^(1/4)                (Eq. 13)
#   - The paper states phi_c ~= 2 V0^(1/4)/sqrt(y) (caption, Table 1),
#     which in our critical_field(kappa=4, M_code, g) = 2*M_code/g means
#     g = sqrt(y) -- i.e. g is NOT a free knob, it's fixed by y.

Mp_GeV = 2.435e18  # reduced Planck mass in GeV, for unit conversion

# Table 1: (phi_P, f, V0^(1/4)[GeV], A, M[GeV], m[GeV], f_c, ns_paper, r_paper, N0_paper)
TABLE1_ROWS = [
    (0.25, 0.65, 3.696e15, 1.164e-13, 5.744e16, 9.04e11,  0.1735,  0.9864, 0.149e-3, 56.30),
    (0.25, 0.70, 3.584e15, 1.282e-13, 5.503e16, 8.74e11,  0.1789,  0.9717, 0.132e-3, 56.29),
    (0.25, 0.75, 3.434e15, 1.336e-13, 5.245e16, 8.29e11,  0.1828,  0.9536, 0.111e-3, 56.25),
    (0.25, 0.80, 3.235e15, 1.296e-13, 4.960e16, 7.63e11,  0.1843,  0.9312, 0.0870e-3, 56.19),
    (1.00, 0.60, 8.873e15, 2.134e-13, 1.278e17, 6.13e12,  0.08907, 0.9930, 5.02e-3,  57.10),
    (1.00, 0.65, 8.603e15, 2.333e-13, 1.225e17, 5.87e12,  0.09252, 0.9768, 4.43e-3,  57.08),
    (1.00, 0.70, 8.262e15, 2.430e-13, 1.171e17, 5.53e12,  0.09521, 0.9575, 3.77e-3,  57.04),
    (1.00, 0.75, 7.836e15, 2.387e-13, 1.113e17, 5.10e12,  0.09696, 0.9346, 3.04e-3,  56.99),
    (2.50, 0.55, 1.482e16, 2.280e-13, 2.116e17, 1.88e13,  0.05409, 0.9921, 43.1e-3,  57.55),
    (2.50, 0.60, 1.443e16, 2.549e-13, 2.032e17, 1.81e13,  0.05665, 0.9783, 38.7e-3,  57.53),
    (2.50, 0.65, 1.394e16, 2.728e-13, 1.947e17, 1.72e13,  0.05880, 0.9619, 33.7e-3,  57.50),
    (2.50, 0.70, 1.334e16, 2.779e-13, 1.859e17, 1.60e13,  0.06046, 0.9427, 28.2e-3,  57.48),
]


def params_from_table_row(row):
    """Convert one Table-1 row into (kappa, M, m, y, phi_c, phi0) in Mp=1
    units, using only relations stated in the paper (see notes above)."""
    phi_P, f, V0_14_GeV, A, M_GeV, m_GeV, f_c, ns_p, r_p, N0_p = row
    V0_14 = V0_14_GeV / Mp_GeV
    m = m_GeV / Mp_GeV
    y = (16.0 * np.pi**2 * A) ** 0.25
    phi0 = phi_P
    phi_c = (f_c / f) * phi0
    kappa = 4.0
    M = V0_14  # since (kappa/4)*M^4 = M^4 = V0 when kappa=4
    return kappa, M, m, y, phi_c, phi0


def module6_validate_table1(verbose=True):
    """Recompute ns, r, N0 at each Table-1 benchmark using our OWN
    V_tree/V_loop/dV_loop/d2V_loop (Module 2 & 4), and compare to the
    paper's published values. Small (~0.5-3%) differences are expected:
    the paper uses next-to-leading-order (Stewart-Lyth) expressions for
    ns/r in Table 1, while epsilon_of_V/eta_of_V here are leading-order
    slow roll -- this is a known, documented approximation, not a bug."""
    results = []
    for row in TABLE1_ROWS:
        phi_P, f, *_ , ns_p, r_p, N0_p = row
        kappa, M, m, y, phi_c, phi0 = params_from_table_row(row)

        dVfun = lambda phi: dV_tree(phi, m) + dV_loop(phi, y, phi_c)
        d2Vfun = lambda phi: m**2 + d2V_loop(phi, y, phi_c)
        Vfun = lambda phi: V_tree(phi, kappa, M, m) + V_loop(phi, y, phi_c)

        eps = epsilon(phi0, Vfun, dVfun)
        et = eta(phi0, Vfun, dVfun, d2Vfun)
        ns = ns_from_eps_eta(eps, et)
        r = r_from_eps(eps)
        N0 = N_efolds(phi0, phi_c, Vfun, dVfun)

        results.append(dict(phi_P=phi_P, f=f, ns_ours=ns, ns_paper=ns_p,
                             r_ours=r, r_paper=r_p, N0_ours=N0, N0_paper=N0_p))
        if verbose:
            print(f"phi_P={phi_P:4.2f} f={f:4.2f}  "
                  f"ns: paper={ns_p:.4f} ours={ns:.4f}   "
                  f"r: paper={r_p:.3e} ours={r:.3e}   "
                  f"N0: paper={N0_p:.2f} ours={N0:.2f}")
    return results

## Stage 6: Reheating and Leptogenesis Estimates

**GOAL:** Estimate the reheat temperature and check consistency with leptogenesis.

**Key Physics:**

The right-handed neutrinos $N$ decay with width:

$$\Gamma_N = \frac{y^2 M_N}{8\pi}$$

The reheat temperature (instant reheating approximation):

$$T_R \approx \left(\frac{90}{\pi^2 g_*}\right)^{1/4} \sqrt{\Gamma_N M_P}$$

For successful leptogenesis, we need $T_R \gtrsim 10^9$ GeV (rough thermal bound).

**The Paper's Unifying Point:** The same Yukawa coupling $y$ that:
1. Shapes the inflationary potential via the loop correction
2. Sets the reheat temperature
3. Sets the leptogenesis scale

In [20]:
#MODULE 7: reheating / leptogenesis order-of-magnitude estimates
 
def decay_width_N(y, M_N):
    """Gamma_N ~ y^2 M_N / (8 pi)."""
    return y**2 * M_N / (8.0 * np.pi)
 
 
def reheat_temperature(Gamma, Mp_GeV=2.435e18, g_star=228.75):
    """TR ~ (90/(pi^2 g*))^{1/4} sqrt(Gamma Mp), a standard instant-reheating
    estimate (up to O(1) factors, as the guide notes)."""
    return (90.0 / (np.pi**2 * g_star))**0.25 * np.sqrt(Gamma * Mp_GeV)
 
 
def module7_reheating(y=1e-3, M_N_GeV=1e13, verbose=True):
    Gamma = decay_width_N(y, M_N_GeV)
    TR = reheat_temperature(Gamma)
    ok_below_MN = TR < M_N_GeV
    ok_leptogenesis = TR > 1e9  # very rough thermal leptogenesis bound
    if verbose:
        print(f"Gamma_N = {Gamma:.3e} GeV")
        print(f"T_R = {TR:.3e} GeV")
        print(f"T_R < M_N ? {ok_below_MN}   T_R >~ 1e9 GeV (leptogenesis)? "
              f"{ok_leptogenesis}")
    return dict(Gamma=Gamma, TR=TR, ok_below_MN=ok_below_MN,
                ok_leptogenesis=ok_leptogenesis)

## Stage 7: Main Execution

Run all modules to verify the reconstruction works.

In [21]:
if __name__ == "__main__":
    print("=== Module 1: slow-roll sanity check ===")
    module1_sanity_check()
    print("\n=== Module 2: critical field ===")
    module2_checkpoint()
    print("\n=== Module 3: tree-level blue tilt ===")
    module3_tree_level_blue_tilt()
    print("\n=== Module 4: hilltop check ===")
    module4_hilltop_check()
    print("\n=== Module 5: full observables ===")
    module5_observables()
    print("\n=== Module 7: reheating estimate ===")
    module7_reheating()
 

=== Module 1: slow-roll sanity check ===
N=50: ns(exact)=0.9604  ns(1-2/N approx)=0.9600  r=0.1584
N=60: ns(exact)=0.9669  ns(1-2/N approx)=0.9667  r=0.1322

=== Module 2: critical field ===
phi_c = 1.000000e-02 Mp  (S=0 valley unstable for phi < phi_c)
m_S^2(phi_c) = 0.000e+00 (should be ~0)

=== Module 3: tree-level blue tilt ===
Tree level at phi=1.000e-02: eps=8.000e-12 eta=4.000e-04 ns=1.0008  (blue tilt if ns>1: True)

=== Module 4: hilltop check ===
Local maximum found: True  at phi ~ 3.8458e-01 Mp

=== Module 5: full observables ===
phi_max(hilltop)=3.8458e-01 Mp  phi_c=6.6731e-02 Mp  phi_end=6.6731e-02 Mp  phi*=2.4993e-01 Mp
eps=9.400e-06  eta=-5.324e-03  ns=0.9893  r=1.504e-04

=== Module 7: reheating estimate ===
Gamma_N = 3.979e+05 GeV
T_R = 4.398e+11 GeV
T_R < M_N ? True   T_R >~ 1e9 GeV (leptogenesis)? True


## Stage 8: Figure Generation

Generate all figures to reproduce the paper's qualitative results.

In [22]:
"""
Generates every figure for the notebook. All functions are already defined
in the previous cells, so no import is needed.
Figures are written to ./figures/.
"""
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq
 
import os
FIGDIR = os.path.join(os.getcwd(), "figures")
os.makedirs(FIGDIR, exist_ok=True)
 
# Note: All functions (epsilon_toy, ns_toy, r_toy, V_tree, critical_field, 
# make_V_and_dV, find_hilltop, V_total, epsilon, eta, ns_from_eps_eta, 
# r_from_eps, N_efolds, module5_observables, etc.) are already defined 
# in the previous code cells!
 
def savefig(fig, name):
    path = os.path.join(FIGDIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("wrote", path)
    plt.close(fig)
 
 
#Figure 1 (Module 1): toy chaotic-inflation sanity check
def fig1_toy_check():
    phi_end = np.sqrt(2.0)
    N = np.linspace(5, 80, 300)
    ns_exact = np.array([ns_toy(n, phi_end) for n in N])
    ns_approx = 1.0 - 2.0 / N
    r_exact = np.array([r_toy(n, phi_end) for n in N])
 
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    ax = axes[0]
    ax.plot(N, ns_exact, label=r"exact: $1-6\epsilon+2\eta$", lw=2.2)
    ax.plot(N, ns_approx, "--", label=r"large-$N$ limit: $1-2/N$", lw=1.8)
    ax.set_xlabel("Number of e-folds before end of inflation, $N$")
    ax.set_ylabel(r"$n_s$")
    ax.set_title(r"Sanity check: $V=\frac{1}{2}m^2\phi^2$")
    ax.legend()
    ax.grid(alpha=0.3)
 
    ax = axes[1]
    ax.plot(N, r_exact, color="firebrick", lw=2.2)
    ax.set_xlabel("Number of e-folds before end of inflation, $N$")
    ax.set_ylabel(r"$r=16\epsilon$")
    ax.set_title("Tensor-to-scalar ratio (toy model)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig1_toy_sanity_check.png")
 
 
#Figure 2 (Module 2): tree-level valley potential V0(phi) = S=0 slice
def fig2_tree_potential(kappa=1.0, M=1e-3, m=1e-8):
    phi_c = critical_field(kappa, M, g=1.0)
    if phi_c == 0:
        return
    phis = np.linspace(0, 1.3 * phi_c, 400)
    V0 = V_tree(phis, kappa, M, m)
    if V0[0] == 0:
        return
 
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    ax.plot(phis / phi_c, V0 / V0[0], lw=2.2, color="darkorange")
    ax.axvline(1.0, color="gray", ls="--", lw=1.2,
               label=r"$\phi_c$ (waterfall trigger)")
    ax.set_xlabel(r"$\phi / \phi_c$")
    ax.set_ylabel(r"$V_0(\phi) / V_0(0)$")
    ax.set_title("Tree-level inflationary valley ($S=0$)\n"
                  r"$V_0=\frac{\kappa}{4}M^4+\frac{1}{2}m^2\phi^2$ -- nearly flat plateau")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig2_tree_level_valley.png")
 
#Figure 3 (Module 3): Tree-level blue tilt problem
def fig3_tree_level_blue_tilt(kappa=1.0, M=1e-2, m=1e-6):
    """Show ns(phi) on V_tree ALONE (no loop correction): this stays
    above 1 for all sub-Planckian phi, illustrating the blue tilt problem."""
 
    def Vfun(phi):
        return V_tree(phi, kappa, M, m)
 
    def dVfun(phi):
        return dV_tree(phi, m)
 
    # Use a log-spaced grid to better resolve the behavior
    phis = np.logspace(-4, -1, 400)  # from 1e-4 to 1e-1
 
    # Compute slow-roll parameters EXACTLY
    eps = np.zeros_like(phis)
    eta_val = np.zeros_like(phis)
    ns = np.zeros_like(phis)
 
    for i, phi in enumerate(phis):
        V = Vfun(phi)
        dV = dVfun(phi)
        d2V = m**2  # For tree-level, V'' = m²
 
        eps[i] = 0.5 * (dV / V)**2  # Mp=1
        eta_val[i] = d2V / V        # Mp=1
        ns[i] = 1.0 - 6.0 * eps[i] + 2.0 * eta_val[i]
 
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
 
    # Plot ns(phi)
    ax.semilogx(phis, ns, lw=2.5, color="steelblue", 
                label=r"$n_s(\phi) = 1 - 6\epsilon + 2\eta$")
 
    # Reference lines
    ax.axhline(1.0, color="gray", ls="--", lw=1.5, label=r"$n_s = 1$ (scale-invariant)")
    ax.axhline(0.963, color="crimson", ls=":", lw=1.5, 
               label=r"WMAP5: $n_s = 0.963 \pm 0.012$")
 
    # Set y-axis limits to clearly show the blue tilt
    ax.set_ylim(0.98, 1.005)
 
    ax.set_xlabel(r"$\phi\ / \ M_P$", fontsize=12)
    ax.set_ylabel(r"$n_s$", fontsize=12)
    ax.set_title("Tree-level hybrid inflation: $n_s > 1$ (blue tilt)\n"
                 r"for all sub-Planckian $\phi$ -- the problem WMAP5 exposed", 
                 fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3, which='both')
 
    # Add annotation showing the problem
    ax.annotate("BLUE TILT\n($n_s > 1$)", 
                xy=(0.005, 1.0004), 
                xytext=(0.002, 1.0025),
                fontsize=10, color='darkblue',
                arrowprops=dict(arrowstyle='->', color='darkblue', lw=1))
 
    ax.annotate("WMAP5 prefers\nRED TILT ($n_s < 1$)", 
                xy=(0.05, 0.963), 
                xytext=(0.025, 0.955),
                fontsize=9, color='crimson',
                arrowprops=dict(arrowstyle='->', color='crimson', lw=1))
 
    fig.tight_layout()
    savefig(fig, "fig3_tree_level_blue_tilt.png")
    return phis, ns
 
#Figure 4 (Module 4): loop correction creates a hilltop maximum
def fig4_hilltop(kappa=1.0, M=1e-3, m=1e-8, y=1e-2, g=1.0):
    phi_c = critical_field(kappa, M, g)
    Vfun, dVfun, d2Vfun = make_V_and_dV(kappa, M, m, y, phi_c)
    phi_max = find_hilltop(Vfun, dVfun, 1e-4, 0.9)
    if phi_max is None:
        return
    phis = np.linspace(1e-5, 1.3 * phi_max, 600)
    Vtree = V_tree(phis, kappa, M, m)
    Vtot = V_total(phis, kappa, M, m, y, phi_c)
 
    fig, ax = plt.subplots(figsize=(6.8, 4.6))
    V0 = V_tree(np.array([0.0]), kappa, M, m)[0]
    if V0 == 0:
        return
    ax.plot(phis, Vtree / V0, "--", lw=1.8, color="gray",
            label=r"tree level $V_0(\phi)$ (monotonic)")
    ax.plot(phis, Vtot / V0, lw=2.4, color="crimson",
            label=r"tree + 1-loop $V_0+V_{\rm loop}$ (hilltop)")
    ax.axvline(phi_max, color="black", ls=":", lw=1.3,
               label=rf"hilltop $\phi_{{\rm max}}={phi_max:.3f}\,M_P$")
    ax.set_xlabel(r"$\phi\ / \ M_P$")
    ax.set_ylabel(r"$V(\phi)\, / \, V(0)$")
    ax.set_title("One-loop Coleman-Weinberg correction from Yukawa-coupled\n"
                  r"right-handed neutrinos generates a hilltop maximum")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig4_hilltop_formation.png")
    return phi_max
 
 
#Figure 5 (Module 5): ns(phi) and N(phi) along the inflaton trajectory
def fig5_ns_profile(kappa=1.0, M=1e-3, m=1e-8, y=1e-2, g=1.0):
    phi_c = critical_field(kappa, M, g)
    Vfun, dVfun, d2Vfun = make_V_and_dV(kappa, M, m, y, phi_c)
    phi_max = find_hilltop(Vfun, dVfun, 1e-4, 0.9)
    if phi_max is None:
        return
    if phi_c >= phi_max:
        return
 
    phis = np.geomspace(phi_c * 1.0005, phi_max * 0.999995, 400)
    Ns, nss, rs = [], [], []
    for p in phis:
        N = N_efolds(p, phi_c, Vfun, dVfun)
        eps = epsilon(p, Vfun, dVfun)
        eta_val = eta(p, Vfun, dVfun, d2Vfun)
        Ns.append(N)
        nss.append(ns_from_eps_eta(eps, eta_val))
        rs.append(r_from_eps(eps))
    Ns, nss, rs = map(np.array, (Ns, nss, rs))
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    ax = axes[0]
    ax.plot(phis, nss, lw=2.0, color="teal")
    ax.axhline(1.0, color="gray", ls="--", lw=1)
    ax.axhline(0.965, color="crimson", ls=":", lw=1.2,
               label=r"WMAP5-era best-fit $n_s\approx0.96$-$0.97$")
    ax.set_xlabel(r"$\phi_*\ / \ M_P$")
    ax.set_ylabel(r"$n_s(\phi_*)$")
    ax.set_title("Spectral index along the trajectory\n"
                  "(blue tilt near phi_c, red tilt approaching the hilltop)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
 
    ax = axes[1]
    ax.plot(phis, Ns, lw=2.0, color="purple")
    ax.axhspan(50, 60, color="gold", alpha=0.25, label="typical $N=50$-$60$ window")
    ax.set_xlabel(r"$\phi_*\ / \ M_P$")
    ax.set_ylabel(r"$N(\phi_*)$ (e-folds to end of inflation)")
    ax.set_title("E-folds pile up logarithmically near the hilltop")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig5_ns_and_N_profile.png")
    return phis, nss, Ns
 
 
#Figure 6 (Module 6): ns-r scatter over the paper's ACTUAL Table 1
# benchmark grid, plotted against the paper's own reported points.
def fig6_ns_r_scan():
    results = module6_validate_table1(verbose=False)
    ns_ours = [r["ns_ours"] for r in results]
    r_ours  = [r["r_ours"]  for r in results]
    ns_pap  = [r["ns_paper"] for r in results]
    r_pap   = [r["r_paper"]  for r in results]

    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    ax.scatter(ns_pap, r_pap, marker="x", s=60, color="black", label="paper (Table 1)")
    ax.scatter(ns_ours, r_ours, marker="o", s=45, color="crimson",
               facecolors="none", label="this notebook")
    ax.set_xlabel(r"$n_s$")
    ax.set_ylabel(r"$r$")
    ax.set_yscale("log")
    ax.set_title("RCHI benchmark points (Table 1): notebook vs. paper")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig6_ns_r_scan.png")
    return results
 
#Figure 7 (Module 6): ns vs f for each phi_P group in Table 1 --
# the paper's own scan variable (compare to their Fig. 5, log10(A) & ns vs f),
# rather than an invented y-grid with no connection to a real solution.
def fig7_ns_vs_phistar():
    results = module6_validate_table1(verbose=False)

    # group by phi_P (0.25, 1.00, 2.50)
    phi_Ps = sorted(set(r["phi_P"] for r in results))
    colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(phi_Ps)))

    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    for phi_P, c in zip(phi_Ps, colors):
        group = [r for r in results if r["phi_P"] == phi_P]
        group.sort(key=lambda r: r["f"])
        fs = [g["f"] for g in group]
        ns_ours = [g["ns_ours"] for g in group]
        ns_pap = [g["ns_paper"] for g in group]
        ax.plot(fs, ns_ours, "-o", color=c, lw=2.0,
                 label=rf"$\phi_P={phi_P:g}$ (this notebook)")
        ax.plot(fs, ns_pap, "--x", color=c, lw=1.2, alpha=0.7,
                 label=rf"$\phi_P={phi_P:g}$ (paper)")

    ax.axhline(1.0, color="gray", ls=":", lw=1)
    ax.set_xlabel(r"$f = \tilde\phi_0/\tilde\phi_M$")
    ax.set_ylabel(r"$n_s$")
    ax.set_title("Spectral index vs. hilltop fraction $f$\n"
                  "(Table 1 benchmarks: notebook vs. paper)")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    savefig(fig, "fig7_ns_vs_phistar_subplanckian.png")
    return results
 
 
# Run all figure generators
if __name__ == "__main__":
    print("Generating figures...")
    fig1_toy_check()
    fig2_tree_potential()
    fig3_tree_level_blue_tilt()
    fig4_hilltop()
    fig5_ns_profile()
    fig6_ns_r_scan()
    fig7_ns_vs_phistar()
    print("All figures generated successfully!")
 

Generating figures...
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig1_toy_sanity_check.png
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig2_tree_level_valley.png
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig3_tree_level_blue_tilt.png
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig4_hilltop_formation.png
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig5_ns_and_N_profile.png
wrote d:\M-C-S\my files (pallas-archive)\SUSY Research\Project SUSY\Reproducing [Hybrid Inflation Revisited in Light of WMAP5]\figures\fig6_ns_r_scan.png
wrote d:\M-C-